In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [2]:
sc.addPyFile('/opt/spark-3.5.1/jars/graphframes-0.8.4-spark3.5-s_2.12.jar')

In [11]:
from graphframes import GraphFrame
from pyspark.sql.functions import col

In [5]:
#Vertix Dataframe
v = spark.createDataFrame([
    ('a', 'Alice', 34,'DS',3000),
    ('b', 'Bob', 36,'ML',2000),
    ('c', 'Charlie', 30,'WebDev',800),
    ('d', 'David', 29,'Programmer',1000),
    ('e', 'Esther', 32,'None',0),
    ('f', 'Fanny', 36,'ML',20000),
    ('g', 'Gabby', 60,'Retired',5000)
], ['id', 'name', 'age' , 'job','salary'])
#Edge Dataframe
e = spark.createDataFrame([
    ('a', 'b', 'friend',2),
    ('b', 'c', 'follow',6),
    ('c', 'b', 'follow',7),
    ('f', 'c', 'follow',0),
    ('e', 'f', 'follow',1),
    ('e', 'd', 'friend',2),
    ('d', 'a', 'friend',5),
    ('a', 'e', 'friend',9)
], ['src', 'dst', 'relationship', 'strength'])
g = GraphFrame(v,e)

In [6]:
# users ages > 30 <-- Subgraph
# relationship 'friend'

In [7]:
g.vertices.show()
g.edges.show()

+---+-------+---+----------+------+
| id|   name|age|       job|salary|
+---+-------+---+----------+------+
|  a|  Alice| 34|        DS|  3000|
|  b|    Bob| 36|        ML|  2000|
|  c|Charlie| 30|    WebDev|   800|
|  d|  David| 29|Programmer|  1000|
|  e| Esther| 32|      None|     0|
|  f|  Fanny| 36|        ML| 20000|
|  g|  Gabby| 60|   Retired|  5000|
+---+-------+---+----------+------+



+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  b|  c|      follow|       6|
|  c|  b|      follow|       7|
|  f|  c|      follow|       0|
|  e|  f|      follow|       1|
|  e|  d|      friend|       2|
|  d|  a|      friend|       5|
|  a|  e|      friend|       9|
+---+---+------------+--------+



In [24]:
# users ages > 30 <-- Subgraph
# relationship 'friend'
_ages = g.filterVertices(col('age')>30)

In [25]:
d_ages.vertices.show()
d_ages.edges.show()

+---+------+---+-------+------+
| id|  name|age|    job|salary|
+---+------+---+-------+------+
|  a| Alice| 34|     DS|  3000|
|  b|   Bob| 36|     ML|  2000|
|  e|Esther| 32|   None|     0|
|  f| Fanny| 36|     ML| 20000|
|  g| Gabby| 60|Retired|  5000|
+---+------+---+-------+------+

+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  a|  e|      friend|       9|
|  e|  f|      follow|       1|
+---+---+------------+--------+



In [27]:
#users relationship = friend
d_friend = g.filterEdges(col('relationship')=='friend')

In [28]:
d_friend.vertices.show() # There is a isolated Edges
d_friend.edges.show()

+---+-------+---+----------+------+
| id|   name|age|       job|salary|
+---+-------+---+----------+------+
|  a|  Alice| 34|        DS|  3000|
|  b|    Bob| 36|        ML|  2000|
|  c|Charlie| 30|    WebDev|   800|
|  d|  David| 29|Programmer|  1000|
|  e| Esther| 32|      None|     0|
|  f|  Fanny| 36|        ML| 20000|
|  g|  Gabby| 60|   Retired|  5000|
+---+-------+---+----------+------+

+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  e|  d|      friend|       2|
|  d|  a|      friend|       5|
|  a|  e|      friend|       9|
+---+---+------------+--------+



In [34]:
df_30Plus = g.filterVertices(col('age')>30).filterEdges(col('relationship')=='friend')

In [35]:
df_30Plus.vertices.show()
df_30Plus.edges.show()

+---+------+---+-------+------+
| id|  name|age|    job|salary|
+---+------+---+-------+------+
|  a| Alice| 34|     DS|  3000|
|  b|   Bob| 36|     ML|  2000|
|  e|Esther| 32|   None|     0|
|  f| Fanny| 36|     ML| 20000|
|  g| Gabby| 60|Retired|  5000|
+---+------+---+-------+------+

+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  a|  e|      friend|       9|
+---+---+------------+--------+



In [39]:
df_final =df_30Plus.dropIsolatedVertices()

In [41]:
df_final.vertices.show()
df_final.edges.show()

+---+------+---+----+------+
| id|  name|age| job|salary|
+---+------+---+----+------+
|  a| Alice| 34|  DS|  3000|
|  b|   Bob| 36|  ML|  2000|
|  e|Esther| 32|None|     0|
+---+------+---+----+------+

+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  a|  e|      friend|       9|
+---+---+------------+--------+



In [42]:
df_final.edges.write.parquet('df_30Plus_friendEdges',mode='overwrite')
df_final.vertices.write.parquet('df_30Plus_friendVert',mode='overwrite')

In [43]:
v = spark.read.parquet('df_30Plus_friendVert/')
e = spark.read.parquet('df_30Plus_friendEdges/')

In [44]:
gf_sub = GraphFrame(v,e)

In [45]:
gf_sub.vertices.show()
gf_sub.edges.show()

+---+------+---+----+------+
| id|  name|age| job|salary|
+---+------+---+----+------+
|  a| Alice| 34|  DS|  3000|
|  b|   Bob| 36|  ML|  2000|
|  e|Esther| 32|None|     0|
+---+------+---+----+------+

+---+---+------------+--------+
|src|dst|relationship|strength|
+---+---+------------+--------+
|  a|  b|      friend|       2|
|  a|  e|      friend|       9|
+---+---+------------+--------+



In [47]:
#filter -- edges & vertices at Same time ! (TRiplet Filter)
#       V ------- E------- V

In [50]:
df = g.find('(v1)-[e1]->(v2)')

In [51]:
df.show(truncate=False)

+--------------------------------+-----------------+--------------------------------+
|v1                              |e1               |v2                              |
+--------------------------------+-----------------+--------------------------------+
|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000}        |
|{f, Fanny, 36, ML, 20000}       |{f, c, follow, 0}|{c, Charlie, 30, WebDev, 800}   |
|{b, Bob, 36, ML, 2000}          |{b, c, follow, 6}|{c, Charlie, 30, WebDev, 800}   |
|{c, Charlie, 30, WebDev, 800}   |{c, b, follow, 7}|{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000}        |{a, b, friend, 2}|{b, Bob, 36, ML, 2000}          |
|{a, Alice, 34, DS, 3000}        |{a, e, friend, 9}|{e, Esther, 32, None, 0}        |
|{e, Esther, 32, None, 0}        |{e, d, friend, 2}|{d, David, 29, Programmer, 1000}|
|{e, Esther, 32, None, 0}        |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}       |
+--------------------------------+-----------------+--

In [53]:
df2 = df.filter(col('v1').age < col('v2').age)

In [54]:
df2.show(truncate =False)

+--------------------------------+-----------------+-------------------------+
|v1                              |e1               |v2                       |
+--------------------------------+-----------------+-------------------------+
|{d, David, 29, Programmer, 1000}|{d, a, friend, 5}|{a, Alice, 34, DS, 3000} |
|{c, Charlie, 30, WebDev, 800}   |{c, b, follow, 7}|{b, Bob, 36, ML, 2000}   |
|{a, Alice, 34, DS, 3000}        |{a, b, friend, 2}|{b, Bob, 36, ML, 2000}   |
|{e, Esther, 32, None, 0}        |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}|
+--------------------------------+-----------------+-------------------------+



In [57]:
df3 =df2.filter(col('e1').relationship =='follow')

In [58]:
df3.show(truncate=False)

+-----------------------------+-----------------+-------------------------+
|v1                           |e1               |v2                       |
+-----------------------------+-----------------+-------------------------+
|{c, Charlie, 30, WebDev, 800}|{c, b, follow, 7}|{b, Bob, 36, ML, 2000}   |
|{e, Esther, 32, None, 0}     |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}|
+-----------------------------+-----------------+-------------------------+



In [62]:
df_sub = g.find('(v1)-[e1]->(v2)').\
filter(col('v1').age < col('v2').age)\
.filter(col('e1').relationship =='follow')

In [68]:
df_sub.show(truncate=False)

+-----------------------------+-----------------+-------------------------+
|v1                           |e1               |v2                       |
+-----------------------------+-----------------+-------------------------+
|{c, Charlie, 30, WebDev, 800}|{c, b, follow, 7}|{b, Bob, 36, ML, 2000}   |
|{e, Esther, 32, None, 0}     |{e, f, follow, 1}|{f, Fanny, 36, ML, 20000}|
+-----------------------------+-----------------+-------------------------+



In [64]:
df_sub.printSchema()

root
 |-- v1: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- job: string (nullable = true)
 |    |-- salary: long (nullable = true)
 |-- e1: struct (nullable = false)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |    |-- relationship: string (nullable = true)
 |    |-- strength: long (nullable = true)
 |-- v2: struct (nullable = false)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- job: string (nullable = true)
 |    |-- salary: long (nullable = true)



In [95]:
# e  = df_sub.select(col('e1').src , col('e1').dst  , col('e1').relationship)
e  = df_sub.select('e1.src' ,'e1.dst', 'e1.relationship')
e.show()

+---+---+------------+
|src|dst|relationship|
+---+---+------------+
|  c|  b|      follow|
|  e|  f|      follow|
+---+---+------------+



In [96]:
# v1 =  df_sub.select(col('v1').id , col('v1')['name'] ,col('v1').age , col('v1').job , col('v1').salary)
v1 = df_sub.select('v1.id','v1.name','v1.age','v1.job','v1.salary')
v1.show()

+---+-------+---+------+------+
| id|   name|age|   job|salary|
+---+-------+---+------+------+
|  c|Charlie| 30|WebDev|   800|
|  e| Esther| 32|  None|     0|
+---+-------+---+------+------+



In [97]:
# v2=  df_sub.select(col('v2').id , col('v2')['name'] ,col('v2').age , col('v2').job , col('v2').salary)
v2 = df_sub.select('v2.id','v2.name','v2.age','v2.job','v2.salary')

v2.show()

+---+-----+---+---+------+
| id| name|age|job|salary|
+---+-----+---+---+------+
|  b|  Bob| 36| ML|  2000|
|  f|Fanny| 36| ML| 20000|
+---+-----+---+---+------+



In [98]:
v = v1.union(v2)

In [99]:
v.show()

+---+-------+---+------+------+
| id|   name|age|   job|salary|
+---+-------+---+------+------+
|  c|Charlie| 30|WebDev|   800|
|  e| Esther| 32|  None|     0|
|  b|    Bob| 36|    ML|  2000|
|  f|  Fanny| 36|    ML| 20000|
+---+-------+---+------+------+



In [100]:
gf_sub = GraphFrame(v,e)